# Cell 1: Clone Repository & Enter

In [1]:
!git clone https://github.com/Bhar07gavi/Cloud_Removal.git
%cd Cloud_Removal

Cloning into 'Cloud_Removal'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 69 (delta 7), reused 67 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 41.16 KiB | 6.86 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Cloud_Removal


# Cell 2: Install Dependencies

In [2]:
!pip install -q -r requirements.txt dagshub dvc[s3]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

# Cell 3: Authenticate with DagsHub (Initializes MLflow & DVC remote credentials)

In [5]:
import dagshub

dagshub.init(
    repo_owner='Bhar07gavi',
    repo_name='Cloud_Removal',
    mlflow=True
)

Initialized MLflow to track repo "Bhar07gavi/Cloud_Removal"

Repository Bhar07gavi/Cloud_Removal initialized!

# Cell 4: Pull Data from DVC (or run preprocess if raw images are downloaded)

```
!dvc remote modify dagshub --local auth basic
!dvc remote modify dagshub --local user YOUR_USERNAME
!dvc remote modify dagshub --local password YOUR_TOKEN
!dvc pull -r dagshub
```

In [6]:
# # Option A: Pull preprocessed data via DVC (if configured with DagsHub)
# !dvc pull -r dagshub

# # Option B: Run preprocessing locally
# import os
# if not os.path.exists('dataset/RICE1/cloud'):
#     print('Dataset not found. Please upload or download the RICE dataset first.')
# else:
#     print('Dataset found! Running preprocessing...')
#     !python -m src.data.preprocess --config configs/dataset_rice.yaml

Fetching
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Querying remote cache:   0% 0/1 [00:00<?, ?files/s]
Querying remote cache:   0% 0/1 [00:00<?, ?files/s{'info': ''}]
                                                               
!
  0% |          |0/? [00:00<?,    ?files/s]
md5: 510db66ef176b7f0fab7e864c0daba8f.dir
Fetching
Building workspace index          |0.00 [00:00,    ?entry/s]
Comparing indexes          |2.00 [00:00, 2.42entry/s]
Applying changes          |0.00 [00:00,     ?file/s]
Everything is up to date.
ERROR: failed to pull data from the cloud - Checkout failed for following targets:
dataset
Is your cache up to date?
<https://error.dvc.org/missing-files>
Dataset not found. Please upload or download the RICE dataset first.


# Cell 5: Verify Smoke Test

In [10]:
!python tests/smoke_test.py

Running Smoke Test on CPU...
  [PASS] Generator forward: torch.Size([2, 3, 256, 256])
  [PASS] Discriminator forward: torch.Size([2, 1, 30, 30])
  [PASS] Backward pass executed successfully.
ALL SMOKE TESTS PASSED!


# Cell 6: Run GPU Training (Logs live loss curves & sample image grids to DagsHub MLflow)

In [12]:
!python -m src.data.preprocess --config configs/dataset_rice.yaml

Discovered total 1236 image pairs from 2 source(s).
Total patches generated: 4944
Saved train split (3952 patches) -> data/splits/rice_combined_train.csv
Saved val split (492 patches) -> data/splits/rice_combined_val.csv
Saved test split (500 patches) -> data/splits/rice_combined_test.csv


In [13]:
!python -m src.training.train --config configs/train_rice.yaml --epochs 50

2026/09/26 15:08:16 INFO mlflow.tracking.fluent: Experiment with name 'rice-cloud-removal-pix2pix' does not exist. Creating a new experiment.
Epoch 1/50 | G: 17.9006 | D: 0.5607 | L1: 0.1654
  * New best model saved (val L1: 0.1654)
Epoch 2/50 | G: 15.0896 | D: 0.5425 | L1: 0.1367
  * New best model saved (val L1: 0.1367)
Epoch 3/50 | G: 13.8997 | D: 0.5711 | L1: 0.1263
  * New best model saved (val L1: 0.1263)
Epoch 4/50 | G: 13.0856 | D: 0.5824 | L1: 0.1189
  * New best model saved (val L1: 0.1189)
Epoch 5/50 | G: 12.5973 | D: 0.5840 | L1: 0.1140
  * New best model saved (val L1: 0.1140)
Epoch 6/50 | G: 12.1134 | D: 0.5818 | L1: 0.1092
  * New best model saved (val L1: 0.1092)
Epoch 7/50 | G: 11.5489 | D: 0.5788 | L1: 0.1033
  * New best model saved (val L1: 0.1033)
Epoch 8/50 | G: 11.3419 | D: 0.5777 | L1: 0.1014
  * New best model saved (val L1: 0.1014)
Epoch 9/50 | G: 10.8708 | D: 0.5882 | L1: 0.0970
  * New best model saved (val L1: 0.0970)
Epoch 10/50 | G: 10.7428 | D: 0.5996 | 